# 01 — Bronze discovery

Discovery and profiling only. **No pipeline logic lives here** — production
transformations belong in importable modules under `src/quantum_lake_student/`
(`faq.md:133`; notebook-ordered execution caps the grade at 60).

Run top to bottom. Clear outputs before committing.

Sections follow `notebooks/README.md`:

1. Bronze object and archive-member inventory
2. Schema/type profile per source
3. Missingness, duplicates, ranges, structural invariants
4. Candidate entities, keys, and relationship cardinalities
5. Evidence for rejected cross-source joins

## 0. Setup

Bronze is read-only. Everything below fetches bytes into memory and never
writes back to the `bronze/` prefix.

In [ ]:
import ast
import hashlib
import io
import re
import zipfile
from collections import defaultdict

import pandas as pd
import yaml

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client
from quantum_lake_student.formats import (
    b8_record_bytes,
    iter_b8_records,
    parse_01_records,
)

settings = Settings.from_environment()
client = minio_client(settings)

BRONZE_KEYS = {
    "qec_syndromes": "bronze/source=qec_syndromes/syndromes_dataset.zip",
    "google_qec": "bronze/source=google_qec/google-surface-code-curated.zip",
    "qasmbench": "bronze/source=qasmbench/qasmbench-qec.zip",
}


def fetch(key: str) -> bytes:
    """Read one Bronze object into memory. Never extracts to disk."""
    response = client.get_object(settings.s3_bucket, key)
    try:
        return response.read()
    finally:
        response.close()
        response.release_conn()


def archive_of(key: str) -> zipfile.ZipFile:
    return zipfile.ZipFile(io.BytesIO(fetch(key)))


print(f"bucket={settings.s3_bucket} endpoint={settings.s3_endpoint}")

## 1. Bronze inventory and hashes

These SHA-256 values go into `results/part1/run.json` and
`results/part1/source_trace.parquet` (`input_sha256`). Record them now —
verifying unchanged Bronze is graded (`rubric.md:18`).

In [ ]:
BRONZE_BYTES = {name: fetch(key) for name, key in BRONZE_KEYS.items()}

inventory = pd.DataFrame(
    [
        {
            "source_name": name,
            "bronze_object": BRONZE_KEYS[name],
            "bytes": len(raw),
            "input_sha256": hashlib.sha256(raw).hexdigest(),
        }
        for name, raw in BRONZE_BYTES.items()
    ]
)
inventory

In [ ]:
# Archive members WITHOUT extracting. Also the place to check for unsafe
# member names -- an unsafe path must stop the run (brief.md:212).
def unsafe(member: str) -> bool:
    return member.startswith("/") or ".." in member.split("/")


rows = []
for name, raw in BRONZE_BYTES.items():
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            rows.append(
                {
                    "source_name": name,
                    "archive_member": info.filename,
                    "uncompressed_bytes": info.file_size,
                    "unsafe_name": unsafe(info.filename),
                }
            )

members = pd.DataFrame(rows)
print(f"total members: {len(members)}  unsafe: {int(members['unsafe_name'].sum())}")
members.groupby("source_name").size().to_frame("members")

## 2. `qec_syndromes` — profile and invariants

### 2a. Documented schema vs actual header

`brief.md:203` requires a check for "documented `label` versus actual `labels`".
The bundled `README.txt` documents `label`; the CSV header says `labels`.
This mismatch is a real finding and belongs in `data_issues.parquet`.

In [ ]:
syndrome_archive = zipfile.ZipFile(io.BytesIO(BRONZE_BYTES["qec_syndromes"]))
syndrome_members = sorted(
    n for n in syndrome_archive.namelist() if n.endswith(".csv")
)

print(syndrome_archive.read("README.txt").decode("utf-8", "replace"))
print("-" * 60)
with syndrome_archive.open(syndrome_members[0]) as handle:
    header = handle.readline().decode().strip()
print(f"actual CSV header: {header!r}")

### 2b. Per-file profile

`physical_fault_rate` comes from the filename — it is not a column
(`silver-tables.md:43`). Each file should reconcile to its `nb-10M` count.

In [ ]:
def fault_rate_of(member: str) -> float:
    return float(re.search(r"pfr-([0-9.]+)_", member).group(1))


frames = {}
profile = []
for member in syndrome_members:
    with syndrome_archive.open(member) as handle:
        frame = pd.read_csv(handle)
    frames[member] = frame
    profile.append(
        {
            "archive_member": member,
            "physical_fault_rate": fault_rate_of(member),
            "rows": len(frame),
            "quantity_sum": int(frame["quantity"].sum()),
            "min_quantity": int(frame["quantity"].min()),
            "nulls": int(frame.isna().sum().sum()),
            "distinct_syndromes": frame["syndromes"].nunique(),
        }
    )

profile = pd.DataFrame(profile)
print(f"TOTAL rows: {profile['rows'].sum():,}  (spec: 75,598)")
print(f"TOTAL weighted: {profile['quantity_sum'].sum():,}  (spec: 70,000,000)")
profile

### 2c. Structural invariants of the `syndromes` string

Required checks (`brief.md:204-206`): four rounds, four binary values per
round, positive `quantity`.

In [ ]:
def parse_syndrome(text: str) -> tuple[int, ...]:
    """Flatten the nested-tuple string to 16 values, round first then check."""
    parsed = ast.literal_eval(text)
    return tuple(bit for round_values in parsed for bit in round_values)


sample_member = syndrome_members[4]
sample = frames[sample_member]

print("raw   :", repr(sample["syndromes"].iloc[0]))
print("parsed:", parse_syndrome(sample["syndromes"].iloc[0]))

shapes = sample["syndromes"].map(
    lambda s: tuple(len(r) for r in ast.literal_eval(s))
)
flat = sample["syndromes"].map(parse_syndrome)
domain = sorted({bit for bits in flat for bit in bits})

print(f"\ndistinct round/check shapes : {shapes.value_counts().to_dict()}")
print(f"distinct flattened lengths  : {sorted(set(flat.map(len)))}")
print(f"bit domain                  : {domain}")
print(f"non-positive quantity rows  : {int((sample['quantity'] <= 0).sum())}")

### 2d. The same syndrome under both labels — valid, not a duplicate

`brief.md:206` calls this out explicitly and `faq.md:78` warns against
deduplicating it. This is what makes `syndromes` alone an invalid key.

In [ ]:
label_column = "labels" if "labels" in sample.columns else "label"

per_syndrome = sample.groupby("syndromes")[label_column].nunique()
both = per_syndrome[per_syndrome > 1]
print(f"{sample_member}")
print(f"  syndromes with BOTH labels: {len(both):,} of {len(per_syndrome):,} distinct")

example = both.index[0]
display(sample[sample["syndromes"] == example])

# Candidate key test: is (syndrome, label) unique within one file?
dupes = sample.duplicated(subset=["syndromes", label_column]).sum()
print(f"\nduplicate (syndrome, label) pairs within the file: {dupes}")
print("-> candidate Silver key: (experiment_id, syndrome_bits, logical_error_label)")

## 3. `google_qec` — companion files and binary contracts

Every directory is a set of **row-aligned** files with no ID column anywhere.
Shot *N* is line *N* of each `.01` and record *N* of each `.b8`. Position is
the only join key, which is why one shot needs several `source_trace` rows
(`faq.md:119`).

Note the `properties.yml` field names: `circuit_detectors`,
`circuit_measurements`, `circuit_sweep_bits` — not the shorter names.

In [ ]:
google_archive = zipfile.ZipFile(io.BytesIO(BRONZE_BYTES["google_qec"]))
experiments = sorted(
    {n.split("/")[0] for n in google_archive.namelist() if "/" in n}
)

properties = {
    experiment: yaml.safe_load(
        google_archive.read(f"{experiment}/properties.yml").decode()
    )
    for experiment in experiments
}
pd.DataFrame(properties).T

In [ ]:
# Required companion files. A MISSING one must stop the run (brief.md:212),
# unlike an invalid record, which goes to data_issues.parquet.
DECODERS = (
    "belief_matching",
    "correlated_matching",
    "pymatching",
    "tensor_network_contraction",
)
REQUIRED = (
    "properties.yml",
    "measurements.b8",
    "sweep.b8",
    "detection_events.b8",
    "obs_flips_actual.01",
    *(f"obs_flips_predicted_by_{d}.01" for d in DECODERS),
)

present = set(google_archive.namelist())
missing = [
    f"{experiment}/{name}"
    for experiment in experiments
    for name in REQUIRED
    if f"{experiment}/{name}" not in present
]
print(f"missing required companion files: {missing or 'none'}")

### 3b. Byte-length, alignment, and padding checks

`brief.md:207-208`. Derive every expected length from `properties.yml`
before decoding anything.

Padding is the subtle one: detectors are 200/600 bits, both divisible by 8,
so detector records have **zero** padding. Measurements are 209 and 625 bits
→ 27 and 79 bytes → **7 padding bits each**. Those trailing bits are not
measurements and must never be read as data.

In [ ]:
checks = []
for experiment in experiments:
    props = properties[experiment]
    shots = int(props["shots"])
    widths = {
        "detection_events.b8": int(props["circuit_detectors"]),
        "measurements.b8": int(props["circuit_measurements"]),
        "sweep.b8": int(props["circuit_sweep_bits"]),
    }
    for filename, bits in widths.items():
        blob = google_archive.read(f"{experiment}/{filename}")
        record_bytes = b8_record_bytes(bits)
        checks.append(
            {
                "experiment_id": experiment,
                "file": filename,
                "bits_per_record": bits,
                "record_bytes": record_bytes,
                "padding_bits": record_bytes * 8 - bits,
                "actual_bytes": len(blob),
                "expected_bytes": shots * record_bytes,
                "ok": len(blob) == shots * record_bytes,
            }
        )
    for filename in ("obs_flips_actual.01",) + tuple(
        f"obs_flips_predicted_by_{d}.01" for d in DECODERS
    ):
        values = parse_01_records(google_archive.read(f"{experiment}/{filename}"))
        checks.append(
            {
                "experiment_id": experiment,
                "file": filename,
                "bits_per_record": 1,
                "record_bytes": None,
                "padding_bits": None,
                "actual_bytes": len(values),
                "expected_bytes": shots,
                "ok": len(values) == shots,
            }
        )

checks = pd.DataFrame(checks)
print(f"failing checks: {int((~checks['ok']).sum())}")
checks[checks["file"].str.endswith(".b8")]

In [ ]:
def nonzero_padding_records(blob: bytes, bits: int) -> int:
    """Count records whose unused high bits in the final byte are set."""
    record_bytes = b8_record_bytes(bits)
    padding = record_bytes * 8 - bits
    if not padding:
        return 0
    mask = (0xFF << (8 - padding)) & 0xFF
    return sum(
        1
        for offset in range(0, len(blob), record_bytes)
        if blob[offset + record_bytes - 1] & mask
    )


for experiment in experiments:
    props = properties[experiment]
    for filename, key in (
        ("detection_events.b8", "circuit_detectors"),
        ("measurements.b8", "circuit_measurements"),
    ):
        blob = google_archive.read(f"{experiment}/{filename}")
        bad = nonzero_padding_records(blob, int(props[key]))
        print(f"{experiment:38s} {filename:20s} non-zero padding records: {bad}")

### 3c. The five concepts, on one shot

Conflating measurements / detector events / actual flip / prediction /
decoder error caps the grade at 70 (`rubric.md:131`). Look at them side by
side once so the distinction is concrete.

In [ ]:
experiment = experiments[0]
props = properties[experiment]
detectors = int(props["circuit_detectors"])
measurements = int(props["circuit_measurements"])
shot = 0

det_bytes = b8_record_bytes(detectors)
meas_bytes = b8_record_bytes(measurements)
det_blob = google_archive.read(f"{experiment}/detection_events.b8")
meas_blob = google_archive.read(f"{experiment}/measurements.b8")

det_row = next(
    iter_b8_records(
        det_blob[shot * det_bytes : (shot + 1) * det_bytes], bits_per_record=detectors
    )
)
meas_row = next(
    iter_b8_records(
        meas_blob[shot * meas_bytes : (shot + 1) * meas_bytes],
        bits_per_record=measurements,
    )
)
actual = parse_01_records(google_archive.read(f"{experiment}/obs_flips_actual.01"))
predictions = {
    d: parse_01_records(
        google_archive.read(f"{experiment}/obs_flips_predicted_by_{d}.01")
    )
    for d in DECODERS
}

print(f"experiment          : {experiment}")
print(f"shot_index          : {shot}")
print(f"1 measurement bits  : {measurements} bits, first 24 -> {meas_row[:24]}")
print(f"2 detector events   : {detectors} bits, {sum(det_row)} fired, first 24 -> {det_row[:24]}")
print(f"3 actual flip       : {actual[shot]}")
for d in DECODERS:
    wrong = predictions[d][shot] != actual[shot]
    print(f"4 {d:28s} predicted {predictions[d][shot]}  5 decoder_error={wrong}")

### 3d. Decoder agreement — the signal Task B depends on

Task B asks whether the four decoders make *complementary* mistakes. The gap
between "any decoder wrong" and "all four wrong" is the room a combined model
has to work with.

**Note the base rate.** `actual_observable_flip` sits near 0.49 in this
release and the supplied decoders land around 0.38–0.45 logical error rate.
That is far from published surface-code performance. Model performance is not
graded, but an unexamined anomaly in the *interpretation* is — record what
you observe and say plainly that the absolute rates are near chance.

In [ ]:
agreement = []
for experiment in experiments:
    actual = parse_01_records(
        google_archive.read(f"{experiment}/obs_flips_actual.01")
    )
    preds = {
        d: parse_01_records(
            google_archive.read(f"{experiment}/obs_flips_predicted_by_{d}.01")
        )
        for d in DECODERS
    }
    row = {
        "experiment_id": experiment,
        "distance": properties[experiment]["distance"],
        "shots": len(actual),
        "actual_flip_rate": sum(actual) / len(actual),
    }
    for d, values in preds.items():
        row[f"ler_{d}"] = sum(
            1 for a, p in zip(actual, values) if a != p
        ) / len(actual)
    row["all_four_wrong"] = sum(
        1 for i, a in enumerate(actual) if all(preds[d][i] != a for d in DECODERS)
    )
    row["any_wrong"] = sum(
        1 for i, a in enumerate(actual) if any(preds[d][i] != a for d in DECODERS)
    )
    row["disagreement_window"] = row["any_wrong"] - row["all_four_wrong"]
    agreement.append(row)

pd.DataFrame(agreement).set_index("experiment_id")

## 4. `qasmbench` — circuit structure, no labels

No shots and no labels. This source exists to make the parity-check
mechanism explicit. **Do not fabricate training rows from it** (`faq.md:57`).

`qec_sm_n5.qasm` is the clearest example and the one analysis question 3 is
about: three data qubits `q[3]`, two ancillas `a[2]`, a `syndrome` gate
definition writing adjacent-pair parities into `syn[2]`, and three
conditional corrections.

In [ ]:
qasm_archive = zipfile.ZipFile(io.BytesIO(BRONZE_BYTES["qasmbench"]))
print(qasm_archive.read("small/qec_sm_n5/qec_sm_n5.qasm").decode())

Three parsing traps that cost marks (`data-sources.md:100`,
`quantum-data-primer.md:198`):

1. `q[0]` and `a[0]` are **different qubits** — register-local indexing.
2. The body of `gate syndrome ...` is a *definition*. Its `cx` lines are not
   executed operations; the single `syndrome q[0],q[1],q[2],a[0],a[1];`
   invocation is.
3. `measure a -> syn;` addresses a whole register — that is **2** executed
   measurements, not 1 statement (`silver-tables.md:113`).

So for `qec_sm_n5`: `measurement_count` counts the expanded `measure a -> syn`
(2) plus `measure q -> c` (3) = 5, and `two_qubit_gate_count` must expand the
invoked gate body (4 `cx`) rather than count the definition lines.

In [ ]:
qasm_rows = []
for member in sorted(n for n in qasm_archive.namelist() if n.endswith(".qasm")):
    body = qasm_archive.read(member).decode()
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    qasm_rows.append(
        {
            "archive_member": member,
            "benchmark_name": member.split("/")[1],
            "variant": "transpiled" if member.endswith("_transpiled.qasm") else "source",
            "lines": len(lines),
            "qreg_decls": sum(1 for l in lines if l.startswith("qreg")),
            "creg_decls": sum(1 for l in lines if l.startswith("creg")),
            "gate_defs": sum(1 for l in lines if l.startswith("gate ")),
            "if_statements": sum(1 for l in lines if l.startswith("if(")),
            "measure_statements": sum(1 for l in lines if l.startswith("measure")),
            "sha256": hashlib.sha256(qasm_archive.read(member)).hexdigest()[:16],
        }
    )

pd.DataFrame(qasm_rows)

## 5. Candidate keys and the rejected cross-source join

`brief.md:198` requires investigating at least one relationship you **reject**,
with preserved evidence. The obvious candidate: can QASMBench circuits be
matched to the Google or simulated experiments?

Both `error_correctiond3_n5` and the Google experiments say "distance 3".
That shared vocabulary is exactly the trap — run the check below and record
the negative result rather than asserting it.

In [ ]:
# Evidence for rejecting a QASMBench <-> experiment join.
google_ids = {
    experiment: {
        "distance": properties[experiment]["distance"],
        "data_qubits": properties[experiment]["data_qubits"],
        "circuit_qubits": properties[experiment]["circuit_qubits"],
        "rounds": properties[experiment]["rounds"],
    }
    for experiment in experiments
}

print("Google experiment parameters:")
for name, values in google_ids.items():
    print(f"  {name}: {values}")

print("\nQASMBench circuits declare (from the qreg lines):")
for member in sorted(n for n in qasm_archive.namelist() if n.endswith(".qasm")):
    body = qasm_archive.read(member).decode()
    regs = re.findall(r"qreg\s+(\w+)\[(\d+)\]", body)
    print(f"  {member}: {regs}")

print(
    "\nEvidence: no shared experiment identifier, no round count, no shot data,\n"
    "and qubit counts (5) do not correspond to the Google layouts (17 / 49).\n"
    "-> REJECT any row-level join. Keep QASMBench as its own entity cluster\n"
    "   in Gold, sharing vocabulary but not keys (brief.md:50, faq.md:47)."
)

## 6. Findings to copy into `DECISIONS.md`

Verified against the release on first run:

| # | Finding | Consequence |
|---|---|---|
| 1 | `README.txt` documents `label`; header is `labels` | first `data_issues.parquet` row, severity=warning, action=accepted |
| 2 | 75,598 rows / 70,000,000 weighted, each file exactly 10,000,000 | `row_counts.json` reconciliation target |
| 3 | All syndromes are shape (4,4,4,4), domain {0,1} | Silver `round_count=4`, `check_count=4` |
| 4 | Same syndrome occurs with both labels | key is (experiment, syndrome, label) — do **not** dedupe |
| 5 | `properties.yml` uses `circuit_detectors` / `circuit_measurements` / `circuit_sweep_bits` | parse these exact names |
| 6 | Detector records: 0 padding bits. Measurement records: 7 | only measurements need a padding check |
| 7 | All companion files present, all lengths reconcile | no run-stopping condition in this release |
| 8 | Actual flip rate ~0.49; decoder LER ~0.38–0.45 | report honestly; near-chance is the observation, not a bug to fix |
| 9 | ~28,000 of 50,000 shots sit in the decoder disagreement window | Task B has real signal to combine |
| 10 | QASMBench: 5 qubits vs Google 17/49, no shared ID | documented rejected relationship |

### Next

Move to `src/quantum_lake_student/stages/prepare_data.py` and build the
syndrome Silver table first — the checks in sections 2b–2d become the
validation rules, and rejected rows become `data_issues.parquet` entries.